# Indexing

## Load Secrets

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## Convert to Markdown, Chunk and Embed

In [2]:
import os
import cohere
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(pymupdf4llm.to_markdown("./data/article.pdf"))

dense = co.embed(
    model="embed-v4.0",
    input_type="search_document",
    embedding_types=["float"],
    texts=chunks,
).embeddings.float

print(f"{len(chunks)} chunks ready, each a {len(dense[0])}-d vector.")

91 chunks ready, each a 1536-d vector.


## Connect to QDrant Cluster

In [5]:
from qdrant_client import QdrantClient

qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)

print(f"Existing collections: {[c.name for c in qdrant.get_collections().collections]}")

Existing collections: ['week0_movies']


## Create a New Collection Schema

In [6]:
from qdrant_client.models import Distance, VectorParams

COLLECTION = "week2_article"

qdrant.recreate_collection(
  collection_name=COLLECTION,
  vectors_config={
    "dense": VectorParams(size=len(dense[0]), distance=Distance.COSINE)
  },
)

print(f"collection '{COLLECTION}' ready.")

C:\Users\Ashutosh Bhadoria\AppData\Local\Temp\ipykernel_17648\3270714514.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


collection 'week2_article' ready.


## Upsert Points in the New Collection

In [7]:
from qdrant_client.models import PointStruct

points = [
  PointStruct(
    id=i,
    vector={"dense": dense[i]},
    payload={"text": chunks[i], "source": "./data/article.pdf"},
  )
  for i in range(len(chunks))
]

qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"Points in collection: {qdrant.get_collection(COLLECTION).points_count}")

Points in collection: 91


## Sanity Check for New Collection

In [9]:
sample, _ = qdrant.scroll(collection_name=COLLECTION, limit=3, with_payload=True, with_vectors=False)

for p in sample:
  snippet= p.payload["text"][:120].replace("\n", " ")
  print(f"[{p.id}] {snippet}...")

[0] ## **Hidden Technical Debt in Agentic Systems**   Issue #02 — Eleven years later, the same diagram is being redrawn   **...
[1] Hey friends! 👋   ## Welcome to **Issue #02** of **The AI Systems Engineer Journey** .   Last week we drew the big map: A...
[2] Today we zoom in on the system that's eating the field: the **Agentic AI System.**   Specifically, I want to do somethin...
